# Numerical Methods for Solving ODEs

A hands-on survey of four classical single-step methods applied to a common test problem and compared against the analytical solution.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## Test Problem

We will use the **scalar linear ODE**

$$\frac{dy}{dt} = \lambda \, y, \qquad y(0) = y_0$$

with $\lambda < 0$ (decay).  This problem is the standard stability benchmark for ODE solvers.

**Analytical solution**

$$y(t) = y_0 \, e^{\lambda t}$$

In [ ]:
# Problem parameters
lamb = -2.0   # decay rate
y0      = 1.0    # initial condition
t0, tf  = 0.0, 4.0
h       = 0.2    # step size

def f(t, y):
    return lamb * y

def analytical(t):
    # TODO: Implement the analytical solution for the ODE
    return 0.0

t_exact = np.linspace(t0, tf, 500)
y_exact = analytical(t_exact)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(t_exact, y_exact, lw=2)
plt.xlabel("t");
plt.ylabel("y(t)")
plt.title(f"Analytical solution  (λ={lamb}, y₀={y0})")
plt.grid(True, alpha=0.3)
plt.tight_layout();
plt.show()

## Helper Utilities

In [ ]:
def make_grid(t0, tf, h):
    return np.arange(t0, tf + h, h)


def plot_solution(results: dict, label_exact="Analytical", title="ODE Methods Comparison"):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    ax.plot(t_exact, y_exact, "k-", lw=2, label=label_exact, zorder=5)
    for label, (t, y) in results.items():
        if len(t): ax.plot(t, y, "o--", ms=4, lw=1.4, label=label)
    ax.set_xlabel("t"); ax.set_ylabel("y(t)")
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    for label, (t, y) in results.items():
        if len(t):
            err = np.abs(y - analytical(t))
            ax.semilogy(t, err + 1e-16, "o--", ms=4, lw=1.4, label=label)
    ax.set_xlabel("t"); ax.set_ylabel("|error|")
    ax.set_title("Global Error (log scale)"); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## 1. Forward (Explicit) Euler Method

### Derivation

Start from the **full Taylor expansion** of $y$ around $t_n$:

$$y(t_n + h) = y(t_n) + h\, y'(t_n) + \frac{h^2}{2}\, y''(t_n) + \frac{h^3}{6}\, y'''(t_n) + \cdots$$

Forward Euler keeps only the first two terms and drops everything from $h^2$ onward:

$$y(t_n + h) \approx y(t_n) + h\, y'(t_n)$$

Substituting $y'(t_n) = f(t_n, y_n)$ gives the update rule:

$$\boxed{y_{n+1} = y_n + h\, f(t_n,\, y_n)}$$

The discarded terms form the **local truncation error**, the mistake made in a single step:

$$\tau_n = \frac{h^2}{2}\, y''(t_n) + \frac{h^3}{6}\, y'''(t_n) + \cdots$$

Dominant (largest) surviving term scales like $h^2$. If you halve $h$, the per-step error shrinks by a factor of ~4.

### Properties

| Property | Value |
|---|---|
| Type | **Explicit** ($y_{n+1}$ solved directly) |
| Local truncation error | $\propto h^2$ |
| Global error | $\propto h$ (**first-order** method) |

In [ ]:
def euler_forward(f, t0, tf, y0, h):
    # TODO: implement

    return np.array([]), np.array([])

t, y = euler_forward(f, t0, tf, y0, h)
plot_solution({"Forward Euler": (t, y)}, title="Forward Euler")

### Stability

Applying the update to $y' = \lambda y$ gives $y_{n+1} = (1 + h\lambda)\,y_n$. Each step multiplies the solution by $R = 1 + h\lambda$. If $|R| > 1$ the numerical solution grows without bound regardless of the true solution and that is instability. Setting $|R| \leq 1$ and solving gives a concrete relationship between $\lambda$ and $h$:

$$|1 + h\lambda| \leq 1 \implies h \leq \frac{2}{|\lambda|}$$

So for a given decay rate $\lambda$, there is a maximum allowable step size. The method is **conditionally stable**.

---
## 2. Backward (Implicit) Euler Method

### Derivation

Start from the same Taylor expansion, but expand **backwards** from $t_{n+1}$:

$$y(t_n) = y(t_{n+1}) - h\, y'(t_{n+1}) + \cdots$$

Rearranging for $y_{n+1}$ and substituting $y'(t_{n+1}) = f(t_{n+1}, y_{n+1})$:

$$\boxed{y_{n+1} \approx y_n + h\, f(t_{n+1},\, y_{n+1})}$$

This looks identical to Forward Euler except $f$ is evaluated at $t_{n+1}$ instead of $t_n$. The catch is that $y_{n+1}$ appears on **both sides**, it is an implicit equation that must be solved at each step.

### Properties

| Property | Value |
|---|---|
| Type | **Implicit** (requires solving an equation for $y_{n+1}$) |
| Local truncation error | $\propto h^2$ |
| Global error | $\propto h$ (**first-order** method) |

In [ ]:
def euler_backward(f, t0, tf, y0, h):
    # TODO: implement

    return np.array([]), np.array([])

t, y = euler_backward(f, t0, tf, y0, h)
plot_solution({"Backward Euler": (t, y)}, title="Backward Euler")

### Stability

Each step multiplies the solution by $R = \dfrac{1}{1 - h\lambda}$. For stability we need $|R| \leq 1$, i.e. $|1 - h\lambda| \geq 1$. When $\lambda < 0$ (decay), the term $-h\lambda$ is positive, so $|1 - h\lambda| = 1 + h|\lambda| > 1$ for **any** $h > 0$ — the condition is always satisfied. Unlike Forward Euler, there is no constraint on step size. This is what **A-stable** means: unconditionally stable for all problems with $\text{Re}(\lambda) < 0$.

---
## 3. Heun's Method (Corrector-Predictor Method)

### Idea

Forward Euler uses only the slope at $t_n$. Heun's method gets a better estimate by also using the slope at $t_{n+1}$, but to evaluate that slope we first need a value there. So the method splits each step into two stages.

**Predictor**: take a cheap Forward Euler step to get a rough value $y^p_{n+1}$:

$$y^p_{n+1} = y_n + f(t_n,\, y_n) \cdot h$$

**Corrector**: compute the correction $\epsilon_n$ from how much the slope changed, then apply it:

$$\epsilon_n = \bigl(f(t_{n+1},\, y^p_{n+1}) - f(t_n,\, y_n)\bigr) \cdot \frac{h}{2}$$

$$\boxed{y^c_{n+1} = y^p_{n+1} + \epsilon_n}$$

The predictor gets us into the right neighbourhood; the corrector refines it using slope information from the end of the interval.

### Properties

| Property | Value |
|---|---|
| Type | **Explicit** — no equation to solve |
| Local truncation error | $\propto h^3$ |
| Global error | $\propto h^2$ — **second-order** method |

In [ ]:
def heun(f, t0, tf, y0, h):
    # TODO: implement

    return np.array([]), np.array([])

t, y = heun(f, t0, tf, y0, h)
plot_solution({"Heun": (t, y)}, title="Heun's Method")

### Stability

Heun is explicit, so it has a finite stability region like Forward Euler. Applying it to $y' = \lambda y$ gives an amplification factor $R = 1 + h\lambda + \frac{(h\lambda)^2}{2}$, which is the first three terms of $e^{h\lambda}$ — a closer match to the true solution than Forward Euler's $R = 1 + h\lambda$. The stability condition $|R| \leq 1$ is satisfied for a larger range of $h$ than Forward Euler, but there is still a maximum step size beyond which the method blows up.

---
## 4. Heun's Method with Adaptive Step Size

### Idea

In fixed-step Heun, $\epsilon_n$ is thrown away after correcting. In the adaptive version we **reuse it as a local error estimate** — if $|\epsilon_n|$ is small we double $h$ for the next step; if it is large we halve $h$.

### Algorithm

At each step compute $\epsilon_n$ as before:

$$y^p_{n+1} = y_n + f(t_n,\, y_n) \cdot h$$

$$\epsilon_n = \bigl(f(t_{n+1},\, y^p_{n+1}) - f(t_n,\, y_n)\bigr) \cdot \frac{h}{2}$$

$$y^c_{n+1} = y^p_{n+1} + \epsilon_n$$

Then adjust $h$ for the next step based on $|\epsilon_n|$ against a threshold $\tau$:

| Condition | Action |
|---|---|
| $\|\epsilon_n\| > \tau$ | Set $h \leftarrow h/2$ |
| $\|\epsilon_n\| < \tau/4$ | Set $h \leftarrow 2h$ |
| otherwise | Keep $h$ |

In [ ]:
def heun_adaptive(f, t0, tf, y0, h0, tau=1e-3):
    # TODO: implement

    return np.array([]), np.array([])

t_ad, y_ad = heun_adaptive(f, t0, tf, y0, h0=h, tau=1e-3)
plot_solution({"Heun adaptive": (t_ad, y_ad)}, title="Heun Adaptive")

---
## 5. Interactive Explorer

Use the sliders to change parameters and see all four methods update in real time.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
%matplotlib inline

out = widgets.Output()

style = {"description_width": "initial"}
lam_slider = widgets.FloatSlider(value=-2.0, min=-20.0, max=-0.1, step=0.1,  description="λ",        continuous_update=False, style=style)
h_slider   = widgets.FloatSlider(value=0.2,  min=0.01,  max=1.0,  step=0.01, description="h (step)", continuous_update=False, style=style)
y0_slider  = widgets.FloatSlider(value=1.0,  min=0.1,   max=5.0,  step=0.1,  description="y₀",       continuous_update=False, style=style)
tf_slider  = widgets.FloatSlider(value=4.0,  min=1.0,   max=10.0, step=0.5,  description="t_final",  continuous_update=False, style=style)
tau_slider = widgets.FloatLogSlider(value=1e-3, base=10, min=-6, max=-1, step=0.5, description="τ (adaptive)", continuous_update=False, style=style)

def update(_=None):
    lam  = lam_slider.value
    h    = min(h_slider.value, tf_slider.value / 2)
    y0   = y0_slider.value
    tf   = tf_slider.value
    tau  = tau_slider.value

    def f_i(t, y): return lam * y
    def exact(t):  return y0 * np.exp(lam * t)

    t_ad, y_ad = heun_adaptive(f_i, 0.0, tf, y0, h0=h, tau=tau)

    results_i = {
        "Forward Euler":       euler_forward(f_i, 0.0, tf, y0, h),
        "Backward Euler":      euler_backward(f_i, 0.0, tf, y0, h),
        "Heun":                heun(f_i, 0.0, tf, y0, h),
        f"Heun adaptive (N={len(t_ad)-1})": (t_ad, y_ad),
    }

    t_ex = np.linspace(0.0, tf, 500)

    with out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        ax = axes[0]
        ax.plot(t_ex, exact(t_ex), "k-", lw=2, label="Analytical", zorder=5)
        for label, (t, y) in results_i.items():
            ax.plot(t, y, "o--", ms=4, lw=1.4, label=label)
        ax.set_xlabel("t"); ax.set_ylabel("y(t)")
        ax.set_title(f"All Methods  (h={h:.3f}, λ={lam}, y₀={y0})")
        ax.legend(); ax.grid(True, alpha=0.3)

        ax = axes[1]
        for label, (t, y) in results_i.items():
            ax.semilogy(t, np.abs(y - exact(t)) + 1e-16, "o--", ms=4, lw=1.4, label=label)
        ax.set_xlabel("t"); ax.set_ylabel("|error|")
        ax.set_title("Global Error (log scale)"); ax.legend(); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        stab = 2.0 / abs(lam)
        status = "STABLE" if h < stab else "UNSTABLE"
        print(f"Forward Euler stability limit: h < {stab:.3f}  →  current h={h:.3f}  [{status}]")

for sl in [lam_slider, h_slider, y0_slider, tf_slider, tau_slider]:
    sl.observe(update, names="value")

display(widgets.VBox([lam_slider, h_slider, y0_slider, tf_slider, tau_slider, out]))
update()